# Tunisia Tourism Flux — Exploratory Data Analysis

**Project:** Forecasting monthly tourism flux across Tunisian governorates (2017–2025)  
**Author:** Ikram Saidane — M1 IA & Data Science, Université Paris-Dauphine Tunis  

---

## Objectifs de l'EDA

1. Comprendre la structure et la qualité des données
2. Analyser la distribution des variables cibles
3. Explorer la saisonnalité et les tendances temporelles
4. Identifier les variables les plus corrélées au flux touristique
5. Visualiser les disparités géographiques entre gouvernorats


## 0. Imports & configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

RANDOM_STATE = 42
print('Configuration prête.')

## 1. Chargement des données

In [ ]:
df = pd.read_csv('../data/ex23_flux_touristique.csv')
df['date'] = pd.to_datetime(df['date'])

print(f'Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'Période    : {df["year"].min()} → {df["year"].max()}')
print(f'Gouvernorats : {df["governorate"].nunique()}')
df.head()

## 2. Vue d'ensemble du dataset

In [ ]:
# Types de données et valeurs manquantes
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'valeurs_uniques': df.nunique(),
    'manquants': df.isnull().sum(),
    '% manquants': (100 * df.isnull().sum() / len(df)).round(2)
})
print(f'Valeurs manquantes totales : {df.isnull().sum().sum()}')
summary

In [ ]:
# Statistiques descriptives des variables numériques clés
key_cols = ['observed_index', 'target_stage1_tabular_score', 'lstm_tourism_next_3m',
            'temperature_c', 'rainfall_mm', 'humidity_pct', 'tourism_pressure_score',
            'ndvi_score', 'population_density']
df[key_cols].describe().T.round(2)

## 3. Analyse des variables cibles

In [ ]:
targets = ['observed_index', 'target_stage1_tabular_score', 'lstm_tourism_next_3m']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

colors = ['#4C72B0', '#55A868', '#C44E52']

for i, (col, color) in enumerate(zip(targets, colors)):
    # Histogramme
    ax = axes[0, i]
    sns.histplot(df[col], bins=40, kde=True, ax=ax, color=color, alpha=0.7)
    ax.set_title(f'Distribution — {col}', fontweight='bold')
    ax.set_xlabel('Valeur')
    skew_val = df[col].skew()
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Moyenne: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='orange', linestyle='--', linewidth=1.5, label=f'Médiane: {df[col].median():.1f}')
    ax.legend(fontsize=8)
    ax.set_ylabel('Fréquence')

    # Boxplot
    ax2 = axes[1, i]
    sns.boxplot(y=df[col], ax=ax2, color=color, width=0.4)
    ax2.set_title(f'Boxplot — skewness: {skew_val:.2f}', fontweight='bold')
    ax2.set_ylabel('Valeur')

plt.suptitle('Analyse des variables cibles', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Observation — Variables cibles

Les trois cibles présentent des distributions relativement symétriques (score entre 0 et 100), centrées autour de 45–50. `lstm_tourism_next_3m` montre une distribution légèrement bimodale, reflétant deux régimes touristiques distincts (basse saison vs haute saison).

## 4. Analyse temporelle — Saisonnalité & tendances

In [ ]:
# Saisonnalité mensuelle
monthly = df.groupby('month')['observed_index'].agg(['mean', 'std']).reset_index()
month_names = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Juin',
               'Juil', 'Août', 'Sep', 'Oct', 'Nov', 'Déc']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart saisonnalité
ax = axes[0]
bars = ax.bar(month_names, monthly['mean'], color='#4C72B0', alpha=0.8, edgecolor='white')
ax.errorbar(month_names, monthly['mean'], yerr=monthly['std'],
            fmt='none', color='black', capsize=4, linewidth=1.5)
ax.set_title('Saisonnalité mensuelle du flux touristique', fontweight='bold')
ax.set_xlabel('Mois')
ax.set_ylabel('Observed Index moyen')
ax.tick_params(axis='x', rotation=45)

# Tendance annuelle
yearly = df.groupby('year')['observed_index'].mean().reset_index()
ax2 = axes[1]
ax2.plot(yearly['year'], yearly['observed_index'], marker='o', color='#C44E52',
         linewidth=2, markersize=8)
ax2.fill_between(yearly['year'], yearly['observed_index'], alpha=0.15, color='#C44E52')
ax2.set_title('Tendance annuelle du flux touristique', fontweight='bold')
ax2.set_xlabel('Année')
ax2.set_ylabel('Observed Index moyen')
ax2.set_xticks(yearly['year'])
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Observation — Saisonnalité

Le flux touristique présente une **saisonnalité marquée** avec un pic estival (juin–août) attendu pour la Tunisie. L'année 2020 montre une chute significative due à la COVID-19, suivie d'une reprise progressive. La tendance 2022–2025 est à la hausse.

In [ ]:
# Heatmap mois × année
pivot = df.pivot_table(index='month', columns='year', values='observed_index', aggfunc='mean')
pivot.index = month_names

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Observed Index moyen'})
ax.set_title('Heatmap flux touristique — Mois × Année', fontweight='bold', fontsize=13)
ax.set_xlabel('Année')
ax.set_ylabel('Mois')
plt.tight_layout()
plt.show()

## 5. Analyse géographique — Disparités par gouvernorat

In [ ]:
gov_stats = df.groupby('governorate').agg(
    mean_index=('observed_index', 'mean'),
    std_index=('observed_index', 'std'),
    mean_temp=('temperature_c', 'mean'),
    mean_tourism_pressure=('tourism_pressure_score', 'mean'),
    n_obs=('observed_index', 'count')
).reset_index().sort_values('mean_index', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Classement gouvernorats
ax = axes[0]
colors_bar = ['#C44E52' if v >= gov_stats['mean_index'].quantile(0.75)
              else '#4C72B0' if v <= gov_stats['mean_index'].quantile(0.25)
              else '#55A868' for v in gov_stats['mean_index']]
bars = ax.barh(gov_stats['governorate'], gov_stats['mean_index'],
               color=colors_bar, edgecolor='white', alpha=0.85)
ax.errorbar(gov_stats['mean_index'], gov_stats['governorate'],
            xerr=gov_stats['std_index'], fmt='none', color='black', capsize=3, linewidth=1)
ax.set_title('Flux touristique moyen par gouvernorat', fontweight='bold')
ax.set_xlabel('Observed Index moyen')
ax.axvline(df['observed_index'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Moyenne nationale: {df["observed_index"].mean():.1f}')
ax.legend()

# Scatter température vs flux
ax2 = axes[1]
scatter = ax2.scatter(gov_stats['mean_temp'], gov_stats['mean_index'],
                      c=gov_stats['mean_tourism_pressure'], cmap='RdYlGn',
                      s=150, alpha=0.85, edgecolors='white', linewidth=1.5)
for _, row in gov_stats.iterrows():
    ax2.annotate(row['governorate'], (row['mean_temp'], row['mean_index']),
                 fontsize=7, ha='left', va='bottom',
                 xytext=(4, 4), textcoords='offset points')
plt.colorbar(scatter, ax=ax2, label='Tourism Pressure Score moyen')
ax2.set_title('Température vs Flux touristique par gouvernorat', fontweight='bold')
ax2.set_xlabel('Température moyenne (°C)')
ax2.set_ylabel('Observed Index moyen')

plt.tight_layout()
plt.show()

## 6. Analyse des corrélations

In [ ]:
# Corrélation avec observed_index
num_cols = df.select_dtypes(include=np.number).columns.tolist()
exclude = ['record_id', 'year', 'month', 'utm32_easting', 'utm32_northing',
           'target_stage1_tabular_score', 'target_stage2_lstm_score', 'lstm_tourism_next_3m']
feature_cols = [c for c in num_cols if c not in exclude + ['observed_index']]

corr_target = df[feature_cols + ['observed_index']].corr()['observed_index'].drop('observed_index').sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors_corr = ['#C44E52' if v < 0 else '#55A868' for v in corr_target]
corr_target.plot(kind='barh', ax=ax, color=colors_corr, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Corrélation des features avec observed_index', fontweight='bold', fontsize=13)
ax.set_xlabel('Corrélation de Pearson')
plt.tight_layout()
plt.show()

print('Top 5 corrélations positives :')
print(corr_target.tail(5).round(3))
print('\nTop 5 corrélations négatives :')
print(corr_target.head(5).round(3))

In [ ]:
# Heatmap des corrélations entre features clés
key_features = ['observed_index', 'tourism_pressure_score', 'temperature_c',
                'rainfall_mm', 'humidity_pct', 'ndvi_score',
                'observed_lag_1', 'observed_lag_3', 'rolling_mean_3']

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(df[key_features].corr(), dtype=bool))
sns.heatmap(df[key_features].corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Corrélation'})
ax.set_title('Heatmap des corrélations entre variables clés', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### Observation — Corrélations

- **Lags et moyennes glissantes** (`observed_lag_1`, `rolling_mean_3`) sont fortement corrélés avec `observed_index` → la série temporelle est auto-corrélée, ce qui justifie l'utilisation de LSTM.
- **`tourism_pressure_score`** est très corrélé avec la cible → c'est la variable la plus informative.
- La **température** montre une corrélation positive modérée : les mois chauds correspondent aux pics touristiques.

## 7. Analyse des variables climatiques

In [ ]:
climate_vars = ['temperature_c', 'rainfall_mm', 'humidity_pct', 'ndvi_score']
month_names = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Juin',
               'Juil', 'Août', 'Sep', 'Oct', 'Nov', 'Déc']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(climate_vars):
    monthly_clim = df.groupby('month')[col].mean()
    ax = axes[i]
    ax.plot(month_names, monthly_clim.values, marker='o', linewidth=2, markersize=7)
    ax.fill_between(month_names, monthly_clim.values, alpha=0.15)
    ax.set_title(f'Saisonnalité — {col}', fontweight='bold')
    ax.set_xlabel('Mois')
    ax.set_ylabel(col)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Variables climatiques — Profil mensuel moyen', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Analyse des lags temporels

In [ ]:
lag_cols = ['observed_lag_1', 'observed_lag_2', 'observed_lag_3',
            'observed_lag_6', 'observed_lag_12']

fig, axes = plt.subplots(1, len(lag_cols), figsize=(18, 4))

for i, lag in enumerate(lag_cols):
    ax = axes[i]
    ax.scatter(df[lag], df['observed_index'], alpha=0.3, s=15, color='#4C72B0')
    # Ligne de régression
    z = np.polyfit(df[lag].dropna(), df.loc[df[lag].notna(), 'observed_index'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[lag].min(), df[lag].max(), 100)
    ax.plot(x_line, p(x_line), 'r-', linewidth=2)
    corr = df[[lag, 'observed_index']].corr().iloc[0, 1]
    ax.set_title(f'{lag}\nr={corr:.2f}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Lag value')
    ax.set_ylabel('observed_index' if i == 0 else '')

plt.suptitle('Corrélation observed_index avec ses lags temporels', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Observation — Lags

- **Lag 1 et 2** ont les corrélations les plus fortes → le flux du mois précédent est le meilleur prédicteur immédiat.
- **Lag 12** montre une corrélation élevée → forte saisonnalité annuelle.
- Ces patterns justifient pleinement l'approche LSTM avec séquences temporelles.

## 9. Synthèse EDA

In [ ]:
print('=== SYNTHÈSE EDA ===')
print(f'Dataset       : {df.shape[0]} observations × {df.shape[1]} variables')
print(f'Période       : {df["year"].min()}–{df["year"].max()} ({df["year"].nunique()} ans)')
print(f'Gouvernorats  : {df["governorate"].nunique()}')
print(f'Valeurs manq. : {df.isnull().sum().sum()} (dataset complet)')
print()
print('Variables cibles :')
for col in ['observed_index', 'target_stage1_tabular_score', 'lstm_tourism_next_3m']:
    print(f'  {col:<40} mean={df[col].mean():.2f}  std={df[col].std():.2f}  skew={df[col].skew():.2f}')
print()
print('Top 3 features corrélées avec observed_index :')
print(corr_target.abs().sort_values(ascending=False).head(3).round(3))
print()
print('Mois avec le flux le plus élevé :', month_names[df.groupby("month")["observed_index"].mean().idxmax() - 1])
print('Gouvernorat avec le flux le plus élevé :', gov_stats.sort_values("mean_index", ascending=False).iloc[0]["governorate"])
print('Gouvernorat avec le flux le plus faible :', gov_stats.sort_values("mean_index").iloc[0]["governorate"])